### Compute QFM for CLC with FFT
from trained models, trained with shot noise (and variance regulation) + evaluation with shotnoise

The QFM can be truncated at frequenices trunc_frequencies = [50] + [i for i in onp.arange(500,5000,500)]+[i for i in onp.arange(5000,10000,2500)] + [10000,12500,15000], in order to see how the truncation influences with MSE

R2 score as well as MSE (and coefficients, if not commented out) are saved

In [ ]:
# Importing necessary packages
import sys
import os
import importlib
from pathlib import Path
import cs_functions_jit as csj
import pennylane as qml
import pennylane.numpy as np

import numpy as onp

import jax
from jax import numpy as jnp
#import optax


jax.config.update("jax_enable_x64", True)

path_base = Path(os.getcwd()) 
# Current path for importing custom functions
sys.path.insert(0, str(path_base / "clc_functions")) 



import input_transform
importlib.reload(input_transform)
from input_transform import inputs_transform, inverse_transform_clc

import qnn_layouts_pennylane
importlib.reload(qnn_layouts_pennylane)
import qnn_layouts_pennylane as pqcs

In [ ]:


# Folder in which to find the test inputs
test_inputs_folder =  'test_data/'


transform_output = True
name_out_transf = ''
if transform_output:
    name_out_transf = '_transformedCLC'

### Interval within which transformed clc should be bounded
bound_output = [0.0, 1.0]

### Upper bound for input transformation
transform_input = True
upperbound = np.pi
upperbound_name = '1p0pi'

batch_size = 100
n_batch_name = str(batch_size)

# Learning rate
learning_rate = 0.001
learning_rate_name = '0p001'

### Kept features
features_kept = ['hus', 'clw', 'cli', 'ta', 'pa', 'hwind']
no_of_features = len(features_kept)

### Architecture specifications
no_qubits = no_of_features


In [ ]:

### PQC architecture layout
# No of shots for circuit evaluation
no_shots = 1000 # or #inf
#No of shots that were used in training 
no_shots_training = 'inf'
#Architecture to be used 'ZZXY' or 'XYZ'
name_arch = 'XYZ'

### PQC architecture layout + optimal params
if name_arch == 'XYZ':
    pqc_layout = pqcs.XYZ_circuit;  name_arch = 'XYZ';  n_enc = 4;  n_dec = 2; 
    if no_shots_training == 'inf':
        best_exp = 1 
    else:
        best_exp = 4
elif name_arch == 'ZZXY':
    pqc_layout = pqcs.ZZXY_circuit;  name_arch = 'ZZXY';  n_enc = 2;  n_dec = 5; 
    if no_shots_training == 'inf':
        best_exp = 6 
    else:
        best_exp = 3

n_enc_name = str(n_enc)
n_dec_name = str(n_dec)

# Load optimal params:
if no_shots_training == 'inf':
    params_folder = 'optimal_params/'
    namefilepars = 'optimal_params'
    name_end = ('_upperbound' + upperbound_name + name_out_transf + '_' + name_arch + '_Nenc' + n_enc_name + 
                '_Ndec' + n_dec_name + '_batch' + n_batch_name + '_lr' + learning_rate_name + '_test' + str(best_exp))
    filename_pars = namefilepars + name_end + '.npy'
else: #varreg, trainign with 1000t
    params_folder = 'optimal_params/'
    filename_pars = 'optimal_params_Nshots'+str(no_shots_training)+'_multinomial_transformedCLC_' + name_arch + '_Nenc' +str(n_enc) + '_Ndec' + str(n_dec) + '_batch100_alpha0p005_iniparam12345_test' + str(best_exp) +'.npy'
    

path_file = os.path.join(params_folder, filename_pars)
opt_params_np = np.load(path_file)
opt_params = jnp.asarray(opt_params_np)


# Load test data

In [ ]:
ind_features = [0,1,2,3,4,6]

### ---------------------------------------------------------------------------------------- ###
## ----------------------------------- Load testing data ------------------------------------ ##
### ---------------------------------------------------------------------------------------- ###

namefilein = 'cirrus_inputs_raw_8features.npy'
namefileout = 'cirrus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cirrus_full = np.load(path_file)
test_inputs_cirrus = test_inputs_cirrus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cirrus = np.load(path_file)
no_testing_data_cirrus = test_inputs_cirrus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cirrus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cirrus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cirrus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cirrus = np.where(III_cirrus)[0]
no_test_samples_to_evaluate_cirrus = np.sum(III_cirrus)
print('No. test samples to evaluate (cirrus): ', no_test_samples_to_evaluate_cirrus)

namefilein = 'cumulus_inputs_raw_8features.npy'
namefileout = 'cumulus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_cumulus_full = np.load(path_file)
test_inputs_cumulus = test_inputs_cumulus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_cumulus = np.load(path_file)
no_testing_data_cumulus = test_inputs_cumulus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_cumulus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_cumulus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_cumulus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_cumulus = np.where(III_cumulus)[0]
no_test_samples_to_evaluate_cumulus = np.sum(III_cumulus)
print('No. test samples to evaluate (cumulus): ', no_test_samples_to_evaluate_cumulus)

namefilein = 'deepconv_inputs_raw_8features.npy'
namefileout = 'deepconv_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_deepconv_full = np.load(path_file)
test_inputs_deepconv = test_inputs_deepconv_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_deepconv = np.load(path_file)
no_testing_data_deepconv = test_inputs_deepconv.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_deepconv[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_deepconv[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_deepconv = np.squeeze(np.logical_not(IIIlowclt))
indsIII_deepconv = np.where(III_deepconv)[0]
no_test_samples_to_evaluate_deepconv = np.sum(III_deepconv)
print('No. test samples to evaluate (deepconv): ', no_test_samples_to_evaluate_deepconv)

namefilein = 'stratus_inputs_raw_8features.npy'
namefileout = 'stratus_outputs_raw_8features.npy'
path_file = os.path.join(test_inputs_folder, namefilein)
test_inputs_stratus_full = np.load(path_file)
test_inputs_stratus = test_inputs_stratus_full[:,ind_features]
path_file = os.path.join(test_inputs_folder, namefileout)
test_outputs_stratus = np.load(path_file)
no_testing_data_stratus = test_inputs_stratus.shape[0]
### Get indices of samples with very low condensate, which are the
### the ones removed from the training data, since for them clc=0 !!!
clw = test_inputs_stratus[:, np.where(np.array([k=='clw' for k in features_kept]))[0][0]]
cli = test_inputs_stratus[:, np.where(np.array([k=='cli' for k in features_kept]))[0][0]]
clt = clw + cli
IIIlowclt = (clt <= 1.0e-08)
III_stratus = np.squeeze(np.logical_not(IIIlowclt))
indsIII_stratus = np.where(III_stratus)[0]
no_test_samples_to_evaluate_stratus = np.sum(III_stratus)
print('No. test samples to evaluate (stratus): ', no_test_samples_to_evaluate_stratus)

### Transform inputs if needed
if transform_input:
    bound_input = [0.0, upperbound]
    bounds = [bound_input for _ in features_kept]
    test_inputs_cirrus_t = inputs_transform(test_inputs_cirrus, features_kept, bounds)
    test_inputs_cumulus_t = inputs_transform(test_inputs_cumulus, features_kept, bounds)
    test_inputs_deepconv_t = inputs_transform(test_inputs_deepconv, features_kept, bounds)
    test_inputs_stratus_t = inputs_transform(test_inputs_stratus, features_kept, bounds)

### Convert testing data to jax numpy arrays
jnp_test_inputs_cirrus = jnp.asarray(test_inputs_cirrus_t)
jnp_test_inputs_cumulus = jnp.asarray(test_inputs_cumulus_t)
jnp_test_inputs_deepconv = jnp.asarray(test_inputs_deepconv_t)
jnp_test_inputs_stratus = jnp.asarray(test_inputs_stratus_t)


In [ ]:

### ---------------------------------------------------------------------------------------- ###
## ---------------------------------- Initialize QNN model ---------------------------------- ##
### ---------------------------------------------------------------------------------------- ###

no_gate_angles = pqcs.no_of_angles_pqc(name_arch, no_qubits, n_enc, n_dec)
print(no_gate_angles)
no_params = no_gate_angles + no_qubits + 1

### Define the quantum device

if no_shots == 'inf':
    dev = qml.device('default.qubit.jax', wires=no_qubits)
else:
    dev = qml.device('default.qubit.jax', wires=no_qubits, shots=no_shots)

    
### Define pqc with measured observables
@qml.qnode(dev, interface="jax")
def qnn_pqc(inputs, pars):
    pqc_layout(inputs, pars, n_enc=n_enc, n_dec=n_dec, wires=dev.wires)
    return [qml.expval(qml.PauliZ(i)) for i in range(no_qubits)]

### Define the QNN model (pqc + postprocessing)
@jax.jit
def model_qnn(params, inputs):
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    print(no_angles)
    no_weights = no_qubits
    no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])
    print(weights)
    print(bias)

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]
    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)



In [ ]:

@jax.jit
def linear_pre(x):
    y = x
    return y

@jax.jit
def linear_pre_inverse(y):
    x = y
    return x


In [ ]:
#Take model and separate it st. input and output dimensions match

@jax.jit
def model_qnn_q(params, inputs): #from Lorenzos Code
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    #no_weights = no_qubits
    #no_bias = 1
    angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    #weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    #bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    #weighted_output = weights[0] * measured_batches[0]
    #for i in range(1,no_qubits):
    #    weighted_output = weighted_output + weights[i] * measured_batches[i]
    #predictions = weighted_output + bias
    return measured_batches#jnp.squeeze(predictions)


@jax.jit
def model_qnn_c(params,measured_batches): #from lorenzos code
    # Split paramter vector in the different components 
    no_angles = no_gate_angles
    no_weights = no_qubits
    no_bias = 1
    #angles = jax.lax.dynamic_slice(params, [0], [no_angles])
    weights = jax.lax.dynamic_slice(params, [no_angles], [no_weights])
    bias = jax.lax.dynamic_slice(params, [no_angles + no_weights], [no_bias])

    # Computation of the quantum circuit in 'measured_batches'
    # 'measured_batches' contains a no_qubits-long list, 
    # where the i-th element is the jnp.array containing the 
    # Z(i) expectation value over the input batch
    #measured_batches = qnn_pqc(inputs, angles)

    # Classical post-processing (weighted avg. + bias)
    weighted_output = weights[0] * measured_batches[0]

    for i in range(1,no_qubits):
        weighted_output = weighted_output + weights[i] * measured_batches[i]
    predictions = weighted_output + bias
    return jnp.squeeze(predictions)

## Compute spectrum

In [ ]:
n0 = n_enc
d = no_qubits 

Of = np.fft.fftfreq(2*n0+1,1/(2*n0+1))
freqs =[Of]*d

grids = np.meshgrid(*freqs,indexing = 'ij')
Omegafft = np.stack(grids,axis = -1)


shape = (2*n0+1,)*d+(d,)


Oflat =Omegafft.reshape(-1,d) #flatten

Xft = 2.*np.pi/(2*n0+1)*Oflat #compute support points




In [ ]:


def post_processing(out_batch_old,opt_params):

        outs_batch_new = np.zeros((out_batch_old.shape[0],))
        batch_size = 100
        no_batches_test = int(np.floor(out_batch_old.shape[0]/ batch_size)) ###<----------!!!
        for kk in range(0, no_batches_test-1):
    
            outs_batch =  model_qnn_c(opt_params,out_batch_old[kk*batch_size:(kk+1)*batch_size,:].T)
    
            outs_batch = np.asarray(outs_batch)
            outs_batch = np.squeeze(outs_batch)
            # Clip values in [0,1]
            outs_batch = np.minimum(outs_batch, 1.0)
            outs_batch = np.maximum(outs_batch, 0.0)
            # If the output has been transformed, re-transform it back
            if transform_output: 
                 outs_batch = inverse_transform_clc(outs_batch)
            outs_batch_new[kk*batch_size:(kk+1)*batch_size] = np.copy(outs_batch)
            
        outs_batch =  model_qnn_c(opt_params,out_batch_old[(no_batches_test-1)*batch_size:,:].T)
    
        outs_batch = np.asarray(outs_batch)
        outs_batch = np.squeeze(outs_batch)
        # Clip values in [0,1]
        outs_batch = np.minimum(outs_batch, 1.0)
        outs_batch = np.maximum(outs_batch, 0.0)
            # If the output has been transformed, re-transform it back
        if transform_output: 
                 outs_batch = inverse_transform_clc(outs_batch)
        outs_batch_new[(no_batches_test-1)*batch_size:] = np.copy(outs_batch)

        return outs_batch_new
    


In [ ]:
# Set of frequencies to which to truncate
trunc_frequencies = [50] + [i for i in onp.arange(500,5000,500)]+[i for i in onp.arange(5000,10000,2500)] + [10000,12500,15000]

In [ ]:
no_exp = 1 # how many times do we repeat the experiments


mse_series_cirrus = np.zeros((no_exp,len(trunc_frequencies)+1))
r2_series_cirrus = np.zeros((no_exp,len(trunc_frequencies)+1))
mse_series_deepconv = np.zeros((no_exp,len(trunc_frequencies)+1))
r2_series_deepconv = np.zeros((no_exp,len(trunc_frequencies)+1))
mse_series_stratus = np.zeros((no_exp,len(trunc_frequencies)+1))
r2_series_stratus = np.zeros((no_exp,len(trunc_frequencies)+1))
mse_series_cumulus = np.zeros((no_exp,len(trunc_frequencies)+1))
r2_series_cumulus=  np.zeros((no_exp,len(trunc_frequencies)+1))
#Cft_flat_series = np.zeros((no_exp,Oflat.shape[0],Oflat.shape[1]),dtype = 'complex') # incase we want to save the coefficiencts; memory expensive
for i in range(no_exp):
            print('Exp:',i)
            
            # Compute Fourier Coefficients with FFT, complex coefficients (exp)
            Cft_np = csj.compute_fourier_coeff_fft(model_qnn_q,Oflat,Xft,shape,opt_params,linear_pre_inverse)
            #Cft_flat_series[i,:,:] = Cft_np.reshape(-1,d)
           
            #Sort Frequencies according to trunc_frequencies, compute real coefficients (cos,sin)
            Omegacutfft,Ccos, Csin = csj.find_largest_indices_real_trunc(shape,Omegafft, Cft_np,d,n0,trunc_frequencies) 
            #evluate the Fourier series on test data set (cirrus)
            mse_exp, r2_exp = csj.eval_trunc_cs(Omegacutfft,Ccos,Csin,[i//2 for i in trunc_frequencies], jnp_test_inputs_cirrus, test_outputs_cirrus,indsIII_cirrus,no_testing_data_cirrus,no_test_samples_to_evaluate_cirrus,post_processing,opt_params)
            mse_series_cirrus[i,0:-1] = mse_exp
            r2_series_cirrus[i,0:-1] = r2_exp

            np.save('mse_series_6f_' + name_arch +'_cirrus_' + str(no_shots) +str(no_shots_training)+'_pub',mse_series_cirrus)
            np.save('r2_series_6f_' + name_arch +'_cirrus_' + str(no_shots) +str(no_shots_training)+'_pub',r2_series_cirrus)
    
            mse_exp, r2_exp =csj.eval_trunc_cs(Omegacutfft,Ccos,Csin, [i//2 for i in trunc_frequencies], jnp_test_inputs_deepconv, test_outputs_deepconv,indsIII_deepconv,no_testing_data_deepconv,no_test_samples_to_evaluate_deepconv,post_processing,opt_params)
            mse_series_deepconv[i,0:-1] = mse_exp
            r2_series_deepconv[i,0:-1] = r2_exp

            np.save('mse_series_6f_' + name_arch +'_deepconv_' + str(no_shots) +str(no_shots_training)+'_pub',mse_series_deepconv)
            np.save('r2_series_6f_' + name_arch +'_deepconv_' + str(no_shots)+str(no_shots_training)+'_pub',r2_series_deepconv)
            print('deepconv done')

            mse_exp, r2_exp =csj.eval_trunc_cs(Omegacutfft,Ccos,Csin, [i//2 for i in trunc_frequencies], jnp_test_inputs_cumulus, test_outputs_cumulus,indsIII_cumulus,no_testing_data_cumulus,no_test_samples_to_evaluate_cumulus, post_processing,opt_params)
            mse_series_cumulus[i,0:-1] = mse_exp
            r2_series_cumulus[i,0:-1] = r2_exp

            np.save('mse_series_6f_' + name_arch +'_cumulus_' + str(no_shots)+str(no_shots_training)+'_pub',mse_series_cumulus)
            np.save('r2_series_6f_' + name_arch +'_cumulus_' + str(no_shots)+str(no_shots_training)+'_pub',r2_series_cumulus)
            print('cumulus done')

            mse_exp, r2_exp =csj.eval_trunc_cs(Omegacutfft,Ccos,Csin, [i//2 for i in trunc_frequencies], jnp_test_inputs_stratus, test_outputs_stratus,indsIII_stratus,no_testing_data_stratus,no_test_samples_to_evaluate_stratus,post_processing,opt_params)
            mse_series_stratus[i,0:-1] = mse_exp
            r2_series_stratus[i,0:-1] = r2_exp


            np.save('mse_series_6f_' + name_arch +'_stratus_' + str(no_shots)+str(no_shots_training)+'_pub'+str(no_exp),mse_series_stratus)
            np.save('r2_series_6f_' + name_arch +'_stratus_' + str(no_shots)+str(no_shots_training)+'_pub', r2_series_stratus)
            #np.save('cft_series_6f_' + name_arch + str(no_shots) +'_varreg'+str(no_shots_training),Cft_flat_series)
            print('done')
            print('cirrus done')
